In [2]:
# --- FIX TENSORFLOW / NUMPY ABI MISMATCH ---
# Pin NumPy to 1.x and keep TF/Keras compatible with it.
%pip install -q --upgrade "numpy<2.0" "tensorflow==2.17.*" "keras==3.5.*" "protobuf<5" "h5py<3.11"





[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# PAD-UFES → Safe(0) vs Risk(1) CSVs — AUTO-DETECT LABEL COLUMN (robust)
from pathlib import Path
import re, pandas as pd, numpy as np
from collections import Counter

# ---- paths (your real ones) ----
PAD = Path("/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/skin-app/skin-app/data/PAD-UFES")
IMAGES = PAD / "Images"
META = PAD / "pad-ufes-20_metadata_2025-12-11.csv"

assert IMAGES.exists(), f"Images folder not found: {IMAGES}"
assert META.exists(), f"Metadata not found: {META}"

# ---- read metadata ----
df = pd.read_csv(META)
print("Columns:", list(df.columns))

# ---- utility: normalize a diagnosis string to our 6-code space ----
def norm_diag(x: str):
    if x is None:
        return None
    s = str(x).upper().strip()
    # direct codes
    if s in {"NEV","SEK","MEL","BCC","SCC","ACK"}:
        return s
    # common synonyms / datasets
    if "MELANOMA" in s: return "MEL"
    if "BASAL" in s or "BCC" in s: return "BCC"
    if "SQUAMOUS" in s or "SCC" in s or "INTRAEPIDERMAL CARCINOMA" in s or "BOWEN" in s: return "SCC"
    if "NEV" in s or "NAEV" in s or "NEVUS" in s or "NAEVUS" in s or s=="NV": return "NEV"
    if "SEBORR" in s or "BKL" in s: return "SEK"  # ISIC "BKL" ≈ seborrheic keratosis-like
    if "ACTINIC" in s or "AKIEC" in s or s == "AK" or "SOLAR KERATOSIS" in s: return "ACK"
    return None

# ---- 1) find candidate label columns by name and content ----
name_pat = re.compile(r"(diag|dx|label|lesion[_\s-]*type|patholog|histolog|clinical|finding)", re.I)
cand_cols = [c for c in df.columns if name_pat.search(c)]
# always consider diagnosis_1..3 if present
for extra in ["diagnosis_1","diagnosis_2","diagnosis_3"]:
    if extra in df.columns and extra not in cand_cols:
        cand_cols.append(extra)

def score_column(col):
    series = df[col].astype(str).replace({"nan": None, "None": None, "": None})
    mapped = series.map(norm_diag)
    hits = mapped.notna().sum()
    return hits

scores = [(c, score_column(c)) for c in cand_cols]
scores.sort(key=lambda x: x[1], reverse=True)
print("Candidate label columns (hits):", scores[:10])

if scores and scores[0][1] > 0:
    label_col = scores[0][0]
    diag_series = df[label_col].astype(str).replace({"nan": None, "None": None, "": None})
else:
    # fallback: try the first non-null across any diagnosis_* if present
    diag_cols = [c for c in ["diagnosis_1","diagnosis_2","diagnosis_3"] if c in df.columns]
    if diag_cols:
        tmp = df[diag_cols].astype(str).replace({"nan": None, "None": None, "": None})
        diag_series = tmp.bfill(axis=1).iloc[:,0]
        label_col = f"coalesced({','.join(diag_cols)})"
    else:
        raise AssertionError("Could not find any diagnosis/label column with recognizable values.")

df["diagnosis"] = diag_series.map(norm_diag)
print(f"Chosen label source: {label_col}")
print("Raw diagnosis value counts (top 10):")
print(diag_series.value_counts(dropna=False).head(10))

before = len(df)
df = df[df["diagnosis"].isin({"NEV","SEK","MEL","BCC","SCC","ACK"})].copy()
print(f"Kept {len(df)} / {before} rows with mapped 6-class diagnoses.")

assert len(df) > 0, "No rows mapped. Inspect 'Raw diagnosis value counts' above; we may need to add another synonym."

# ---- 2) resolve images by isic_id ----
isic_col = "isic_id" if "isic_id" in df.columns else None
assert isic_col, "Missing isic_id column; required to match filenames."

# index filenames by stem
stems = {}
for ext in ("*.png","*.jpg","*.jpeg","*.PNG","*.JPG","*.JPEG"):
    for p in IMAGES.rglob(ext):
        stems[p.stem] = str(p)
print("Indexed file stems:", len(stems))

df["img_stem"] = (
    df[isic_col].astype(str)
      .str.replace(r"\.(png|jpg|jpeg)$","", regex=True, case=False)
      .str.strip()
)

def resolve(stem):
    return stems.get(stem) or stems.get(stem.upper()) or stems.get(stem.lower())

df["image_fullpath"] = df["img_stem"].map(resolve)
miss = df["image_fullpath"].isna().sum()
print(f"Matched images: {len(df)-miss} / {len(df)} (missing {miss})")
assert (len(df)-miss) > 0, "Still no filename matches. Do your files look like ISIC_xxxxxxx.* under Images/?"

df = df[~df["image_fullpath"].isna()].copy()

# ---- 3) binary mapping + grouping ----
SAFE = {"NEV","SEK"}         # 0
RISK = {"MEL","BCC","SCC","ACK"}  # 1

df["binary_label"] = df["diagnosis"].map(lambda d: 1 if d in RISK else 0)
df["binary_label_name"] = df["binary_label"].map({0:"safe",1:"risk"})
df["dataset_name"] = "padufes"
df["image_relpath"] = df["image_fullpath"].apply(lambda p: str(Path(p).relative_to(PAD)))

group_col = "patient_id" if "patient_id" in df.columns else ("lesion_id" if "lesion_id" in df.columns else None)
if group_col:
    df["group_id"] = df[group_col].astype(str)
else:
    df["group_id"] = df["diagnosis"] + "_" + df["img_stem"].str[:6]

# ---- 4) patient-grouped stratified 80/20 split ----
VAL_FRACTION = 0.20
rng = np.random.default_rng(42)
train_mask = np.zeros(len(df), dtype=bool)
val_mask   = np.zeros(len(df), dtype=bool)

for y in [0,1]:
    sub = df[df["binary_label"]==y]
    groups = sub["group_id"].unique().tolist()
    rng.shuffle(groups)
    target_val = int(np.round(len(sub) * VAL_FRACTION))
    picked, count = [], 0
    for g in groups:
        n = (sub["group_id"]==g).sum()
        if count < target_val:
            picked.append(g); count += n
    val_idx = df.index[(df["binary_label"]==y) & (df["group_id"].isin(picked))]
    tr_idx  = df.index[(df["binary_label"]==y) & (~df["group_id"].isin(picked))]
    val_mask[val_idx] = True
    train_mask[tr_idx] = True

train_df = df[train_mask].copy()
val_df   = df[val_mask].copy()

# ---- 5) save + report ----
cols_out = ["dataset_name","diagnosis","binary_label","binary_label_name","image_relpath","group_id"]
(train_df[cols_out]).to_csv(PAD/"lesion_binary_train.csv", index=False)
(val_df[cols_out]).to_csv(PAD/"lesion_binary_val.csv", index=False)

def summary(frame, name):
    c = frame["binary_label_name"].value_counts().to_dict()
    return f"{name}: total={len(frame)} | safe={c.get('safe',0)} | risk={c.get('risk',0)}"

print("✔ Wrote:")
print("  -", PAD/"lesion_binary_train.csv")
print("  -", PAD/"lesion_binary_val.csv")
print(summary(train_df, "Train"))
print(summary(val_df,   "Val"))

ctr = Counter(train_df["binary_label"].tolist())
max_n = max(ctr.values())
class_weight = {k: round(max_n/n, 3) for k, n in ctr.items()}
print("Suggested class_weight:", class_weight)


Columns: ['isic_id', 'attribution', 'copyright_license', 'age_approx', 'anatom_site_general', 'anatom_site_special', 'clin_size_long_diam_mm', 'diagnosis_1', 'diagnosis_2', 'diagnosis_3', 'diagnosis_confirm_type', 'fitzpatrick_skin_type', 'image_type', 'lesion_id', 'patient_id', 'sex']
Candidate label columns (hits): [('diagnosis_3', 2298), ('diagnosis_2', 52), ('diagnosis_1', 0), ('diagnosis_confirm_type', 0)]
Chosen label source: diagnosis_3
Raw diagnosis value counts (top 10):
diagnosis_3
Basal cell carcinoma            845
Solar or actinic keratosis      730
Nevus                           244
Seborrheic keratosis            235
Squamous cell carcinoma, NOS    192
Melanoma, NOS                    52
Name: count, dtype: int64
Kept 2298 / 2298 rows with mapped 6-class diagnoses.
Indexed file stems: 2298
Matched images: 2298 / 2298 (missing 0)
✔ Wrote:
  - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/skin-app/skin-app/data/PAD-UFES/lesion_binary_trai

In [4]:
PAD = Path("/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/skin-app/skin-app/data/PAD-UFES")


In [5]:
import os, pandas as pd, numpy as np, tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

# --- CONFIG ---
DATASET_ROOT = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/skin-app/skin-app/data/PAD-UFES"
TRAIN_CSV = os.path.join(DATASET_ROOT, "lesion_binary_train.csv")
VAL_CSV   = os.path.join(DATASET_ROOT, "lesion_binary_val.csv")
ARTIFACTS_DIR = "data/artifacts"
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_HEAD = 12  # a touch longer for stability

os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# --- Load data ---
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)

def fix_path(rel_path):
    p = str(rel_path)
    return p if p.startswith(DATASET_ROOT) else os.path.join(DATASET_ROOT, p)

train_df["fullpath"] = train_df["image_relpath"].apply(fix_path)
val_df["fullpath"]   = val_df["image_relpath"].apply(fix_path)

# --- Class weights (computed, not hard-coded) ---
from collections import Counter
ctr = Counter(train_df["binary_label"].tolist())
maj = max(ctr.values())
class_weights = {k: maj/v for k,v in ctr.items()}
print("class_weights:", class_weights)

# --- tf.data with GENTLER augmentation (pre-batch) ---
AUTOTUNE = tf.data.AUTOTUNE
aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.05),
], name="gentle_aug")

def load_img(path, label):
    b = tf.io.read_file(path)
    x = tf.image.decode_image(b, channels=3, expand_animations=False)
    x = tf.image.resize(x, (IMG_SIZE, IMG_SIZE), antialias=True)
    x = tf.cast(x, tf.float32)  # keep 0–255; model rescales internally
    return x, tf.cast(label, tf.float32)

def make_ds(df, training=True):
    paths = df["fullpath"].values
    labels = df["binary_label"].values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(df), reshuffle_each_iteration=True)
    ds = ds.map(load_img, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x,y: (aug(x, training=True), y), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(train_df, training=True)
val_ds   = make_ds(val_df,   training=False)

# --- Model (MNv3Small; rescale inside the graph; backbone frozen) ---
def build_model():
    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1./127.5, offset=-1, name="mnv3_rescale")(inp)  # [-1,1]
    base = tf.keras.applications.MobileNetV3Small(
        include_top=False, weights="imagenet", pooling="avg", input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    # Keep the default layer name (e.g., "MobileNetV3Small"); we’ll retrieve it later.
    base.trainable = False
    x = base(x, training=False)
    x = layers.Dropout(0.30)(x)
    out = layers.Dense(1, activation="sigmoid", name="risk")(x)
    model = models.Model(inp, out, name="Lesion_Safety_Net")
    return model

model = build_model()
model.compile(
    optimizer=optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)

ckpt1 = callbacks.ModelCheckpoint(
    os.path.join(ARTIFACTS_DIR, "lesion_safety_stage1.keras"),
    monitor="val_auc", mode="max", save_best_only=True, verbose=1
)
es1 = callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=6, restore_best_weights=True, verbose=1)

hist1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    class_weight=class_weights,
    callbacks=[ckpt1, es1],
    verbose=1,
)
print("Stage-1 complete.")


2025-12-15 09:45:14.985610: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-15 09:45:15.026791: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-15 09:45:15.065977: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-15 09:45:15.076046: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-15 09:45:15.122511: I tensorflow/core/platform/cpu_feature_guar

class_weights: {1: 1.0, 0: 3.7963446475195823}


I0000 00:00:1765791917.904559  116260 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1765791917.963844  116260 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1765791917.963909  116260 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1765791917.965873  116260 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1765791917.965942  116260 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:0

Epoch 1/12


I0000 00:00:1765791924.037106  116375 service.cc:146] XLA service 0x7a6e54003740 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1765791924.037353  116375 service.cc:154]   StreamExecutor device (0): NVIDIA GeForce RTX 3070, Compute Capability 8.6
2025-12-15 09:45:24.260500: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-12-15 09:45:25.779400: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 8906


 8/58 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1914 - auc: 0.5247 - loss: 1.1866 - precision: 0.5000 - recall: 0.0108        

I0000 00:00:1765791932.351977  116375 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step - accuracy: 0.4140 - auc: 0.4887 - loss: 1.1308 - precision: 0.7791 - recall: 0.3624
Epoch 1: val_auc improved from -inf to 0.63088, saving model to data/artifacts/lesion_safety_stage1.keras
58/58 ━━━━━━━━━━━━━━━━━━━━ 29s 286ms/step - accuracy: 0.4150 - auc: 0.4886 - loss: 1.1309 - precision: 0.7793 - recall: 0.3640 - val_accuracy: 0.2082 - val_auc: 0.6309 - val_loss: 0.7473 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/12
57/58 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.4335 - auc: 0.4722 - loss: 1.0991 - precision: 0.7782 - recall: 0.4095
Epoch 2: val_auc improved from 0.63088 to 0.64140, saving model to data/artifacts/lesion_safety_stage1.keras
58/58 ━━━━━━━━━━━━━━━━━━━━ 5s 90ms/step - accuracy: 0.4353 - auc: 0.4728 - loss: 1.0994 - precision: 0.7783 - recall: 0.4120 - val_accuracy: 0.7918 - val_auc: 0.6414 - val_loss: 0.6730 - val_precision: 0.7918 - val_recall: 1.0000
Epoch 3/12
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - 

In [6]:
# Stage-2 fine-tuning (gentle), starting from your current `model`
import tensorflow as tf
from tensorflow.keras import layers, callbacks, optimizers

EPOCHS_FINE = 20
CLASS_WEIGHTS = class_weights  # reuse from Stage-1

# Find the application backbone reliably
try:
    backbone = model.get_layer("MobileNetV3Small")
except ValueError:
    backbone = next(L for L in model.layers if isinstance(L, tf.keras.Model))

# Freeze all BatchNorms; unfreeze the rest
for L in backbone.layers:
    if isinstance(L, layers.BatchNormalization):
        L.trainable = False
    else:
        L.trainable = True

# Optionally re-freeze everything except the last ~100 layers
if len(backbone.layers) > 100:
    for L in backbone.layers[:-100]:
        L.trainable = False

model.compile(
    optimizer=optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)

ckpt2 = callbacks.ModelCheckpoint(
    "data/artifacts/lesion_safety_mnv3.keras",
    monitor="val_auc", mode="max", save_best_only=True, verbose=1
)
es2 = callbacks.EarlyStopping(
    monitor="val_auc", mode="max", patience=6, restore_best_weights=True, verbose=1
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE,
    class_weight=CLASS_WEIGHTS,
    callbacks=[ckpt2, es2],
    verbose=1,
)

print("Stage-2 complete.")


Epoch 1/20


58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 262ms/step - accuracy: 0.6051 - auc: 0.4771 - loss: 1.0917 - precision: 0.8016 - recall: 0.6728
Epoch 1: val_auc improved from -inf to 0.65218, saving model to data/artifacts/lesion_safety_mnv3.keras
58/58 ━━━━━━━━━━━━━━━━━━━━ 38s 356ms/step - accuracy: 0.6036 - auc: 0.4774 - loss: 1.0920 - precision: 0.8015 - recall: 0.6702 - val_accuracy: 0.7918 - val_auc: 0.6522 - val_loss: 0.6678 - val_precision: 0.7918 - val_recall: 1.0000
Epoch 2/20
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.5656 - auc: 0.4757 - loss: 1.1319 - precision: 0.7708 - recall: 0.6331
Epoch 2: val_auc did not improve from 0.65218
58/58 ━━━━━━━━━━━━━━━━━━━━ 5s 85ms/step - accuracy: 0.5650 - auc: 0.4759 - loss: 1.1314 - precision: 0.7710 - recall: 0.6318 - val_accuracy: 0.2993 - val_auc: 0.6306 - val_loss: 0.6945 - val_precision: 0.8750 - val_recall: 0.1342
Epoch 3/20
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.4560 - auc: 0.4820 - loss: 1.1169 - precision: 0.7752 - recal

In [7]:
import numpy as np
from sklearn.metrics import confusion_matrix, roc_auc_score

# Collect val probs/labels
val_probs = np.concatenate([model.predict(x, verbose=0) for x, _ in val_ds]).ravel()
val_labels = np.concatenate([y.numpy() for _, y in val_ds]).astype(int)

print("Val AUROC:", roc_auc_score(val_labels, val_probs))

def metrics_at(th):
    pred = (val_probs >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(val_labels, pred, labels=[0,1]).ravel()
    sens = tp / (tp + fn + 1e-9)     # Risk recall
    spec = tn / (tn + fp + 1e-9)
    prec = tp / (tp + fp + 1e-9)
    acc  = (tp + tn) / (tp + tn + fp + fn)
    return dict(th=th, tp=tp, fn=fn, fp=fp, tn=tn, sens=sens, spec=spec, prec=prec, acc=acc)

grid = [metrics_at(t) for t in np.linspace(0.05, 0.99, 95)]
cands = [r for r in grid if r["sens"] >= 0.99]
cands_sorted = sorted(cands, key=lambda r: r["th"]) if cands else sorted(grid, key=lambda r: -r["sens"])

print("Top options (first 5):")
for r in cands_sorted[:5]:
    print({k:(round(v,4) if isinstance(v,float) else v) for k,v in r.items()})
best = cands_sorted[0]
best


2025-12-15 09:47:58.647699: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Val AUROC: 0.6693207762557077
Top options (first 5):
{'th': 0.05, 'tp': 365, 'fn': 0, 'fp': 96, 'tn': 0, 'sens': 1.0, 'spec': 0.0, 'prec': 0.7918, 'acc': 0.7918}
{'th': 0.06, 'tp': 365, 'fn': 0, 'fp': 96, 'tn': 0, 'sens': 1.0, 'spec': 0.0, 'prec': 0.7918, 'acc': 0.7918}
{'th': 0.07, 'tp': 365, 'fn': 0, 'fp': 96, 'tn': 0, 'sens': 1.0, 'spec': 0.0, 'prec': 0.7918, 'acc': 0.7918}
{'th': 0.08, 'tp': 365, 'fn': 0, 'fp': 96, 'tn': 0, 'sens': 1.0, 'spec': 0.0, 'prec': 0.7918, 'acc': 0.7918}
{'th': 0.09, 'tp': 365, 'fn': 0, 'fp': 96, 'tn': 0, 'sens': 1.0, 'spec': 0.0, 'prec': 0.7918, 'acc': 0.7918}


2025-12-15 09:47:59.128766: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


{'th': 0.05,
 'tp': 365,
 'fn': 0,
 'fp': 96,
 'tn': 0,
 'sens': 0.9999999999972603,
 'spec': 0.0,
 'prec': 0.7917570498898227,
 'acc': 0.7917570498915402}

In [8]:
!pip install scikit-learn


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [9]:
# Balance the training set (Safe oversampled to match Risk)
import numpy as np, pandas as pd

# Reuse these from earlier session:
# - train_df, val_df, fix_path, make_ds, aug already defined
safe_df = train_df[train_df.binary_label==0]
risk_df = train_df[train_df.binary_label==1]

n_risk = len(risk_df)
safe_upsampled = safe_df.sample(n=n_risk, replace=True, random_state=42)

train_bal_df = pd.concat([risk_df, safe_upsampled], ignore_index=True).sample(frac=1, random_state=42)
train_bal_df["fullpath"] = train_bal_df["image_relpath"].apply(fix_path)

train_bal_ds = make_ds(train_bal_df, training=True)  # same pipeline/aug as before
print("Balanced train size:", len(train_bal_df), "| Safe:", (train_bal_df.binary_label==0).sum(), "| Risk:", (train_bal_df.binary_label==1).sum())


Balanced train size: 2908 | Safe: 1454 | Risk: 1454


In [10]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
import os
ARTIFACTS_DIR = "data/artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# Weighted focal loss — emphasize Safe errors (alpha_safe > alpha_risk)
def weighted_focal_loss(alpha_safe=3.0, alpha_risk=1.0, gamma=2.0):
    a0 = tf.constant(alpha_safe, tf.float32)
    a1 = tf.constant(alpha_risk, tf.float32)
    g  = tf.constant(gamma, tf.float32)
    eps = 1e-7
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, eps, 1.0-eps)
        pt = tf.where(tf.equal(y_true,1.0), y_pred, 1.0 - y_pred)
        at = y_true * a1 + (1.0 - y_true) * a0
        return tf.reduce_mean(-at * tf.pow(1.0 - pt, g) * tf.math.log(pt))
    return loss

# Build fresh model (same backbone), slightly higher dropout in head
IMG_SIZE = 224
def build_model():
    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1./127.5, offset=-1, name="mnv3_rescale")(inp)
    base = tf.keras.applications.MobileNetV3Small(include_top=False, weights="imagenet", pooling="avg",
                                                  input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    x = base(x, training=False)
    x = layers.Dropout(0.40)(x)  # a touch stronger
    out = layers.Dense(1, activation="sigmoid", name="risk",
                       bias_initializer=tf.keras.initializers.Constant(0.0))(x)  # 0.0 ~ 50/50 prior
    return models.Model(inp, out, name="Lesion_Safety_Net")

model = build_model()

METRICS = [
    tf.keras.metrics.BinaryAccuracy(name="accuracy"),
    tf.keras.metrics.Recall(name="recall"),
    tf.keras.metrics.Precision(name="precision"),
    tf.keras.metrics.AUC(name="auc"),
]

# ---- Stage 1: head warmup on BALANCED data; no class_weight ----
model.compile(optimizer=optimizers.Adam(1e-3),
              loss=weighted_focal_loss(alpha_safe=3.0, alpha_risk=1.0, gamma=2.0),
              metrics=METRICS)

ckpt1 = callbacks.ModelCheckpoint(
    os.path.join(ARTIFACTS_DIR, "lesion_safety_bal_focal_stage1.keras"),
    monitor="val_auc", mode="max", save_best_only=True, verbose=1
)
es1 = callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=6, restore_best_weights=True, verbose=1)

hist1 = model.fit(
    train_bal_ds, validation_data=val_ds, epochs=12,
    callbacks=[ckpt1, es1], verbose=1
)

# ---- Stage 2: fine-tune top layers; keep BatchNorm frozen ----
try:
    backbone = model.get_layer("MobileNetV3Small")
except ValueError:
    backbone = next(L for L in model.layers if isinstance(L, tf.keras.Model))

for L in backbone.layers:
    if isinstance(L, layers.BatchNormalization):
        L.trainable = False
    else:
        L.trainable = True

# Unfreeze only top ~100 layers if present
if len(backbone.layers) > 100:
    for L in backbone.layers[:-100]:
        L.trainable = False

model.compile(optimizer=optimizers.Adam(3e-5),  # lower LR for stability
              loss=weighted_focal_loss(alpha_safe=3.0, alpha_risk=1.0, gamma=2.0),
              metrics=METRICS)

ckpt2 = callbacks.ModelCheckpoint(
    os.path.join(ARTIFACTS_DIR, "lesion_safety_bal_focal_stage2.keras"),
    monitor="val_auc", mode="max", save_best_only=True, verbose=1
)
es2 = callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=6, restore_best_weights=True, verbose=1)

hist2 = model.fit(
    train_bal_ds, validation_data=val_ds, epochs=20,
    callbacks=[ckpt2, es2], verbose=1
)

print("✅ Balanced+Focal training complete.")


Epoch 1/12
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step - accuracy: 0.4942 - auc: 0.4995 - loss: 0.3500 - precision: 0.5066 - recall: 0.2002
Epoch 1: val_auc improved from -inf to 0.53465, saving model to data/artifacts/lesion_safety_bal_focal_stage1.keras
91/91 ━━━━━━━━━━━━━━━━━━━━ 26s 205ms/step - accuracy: 0.4942 - auc: 0.4996 - loss: 0.3498 - precision: 0.5066 - recall: 0.1993 - val_accuracy: 0.2082 - val_auc: 0.5346 - val_loss: 0.3423 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/12
90/91 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.4845 - auc: 0.5043 - loss: 0.3100 - precision: 0.4695 - recall: 0.0510
Epoch 2: val_auc did not improve from 0.53465
91/91 ━━━━━━━━━━━━━━━━━━━━ 7s 80ms/step - accuracy: 0.4848 - auc: 0.5046 - loss: 0.3098 - precision: 0.4700 - recall: 0.0507 - val_accuracy: 0.2082 - val_auc: 0.5224 - val_loss: 0.2995 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 3/12
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.5026 - auc: 0.4797 - l

In [11]:
import numpy as np
from sklearn.metrics import confusion_matrix, roc_auc_score

val_probs = np.concatenate([model.predict(x, verbose=0) for x, _ in val_ds]).ravel()
val_labels = np.concatenate([y.numpy() for _, y in val_ds]).astype(int)

print("Val AUROC:", roc_auc_score(val_labels, val_probs))

def metrics_at(th):
    pred = (val_probs >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(val_labels, pred, labels=[0,1]).ravel()
    sens = tp / (tp + fn + 1e-9)
    spec = tn / (tn + fp + 1e-9)
    prec = tp / (tp + fp + 1e-9)
    acc  = (tp + tn) / (tp + tn + fp + fn)
    return dict(th=th, tp=tp, fn=fn, fp=fp, tn=tn, sens=sens, spec=spec, prec=prec, acc=acc)

grid = [metrics_at(t) for t in np.linspace(0.05, 0.99, 95)]
cands = [r for r in grid if r["sens"] >= 0.99]
print("Top (τ with sens≥0.99):", [{k:(round(v,4) if isinstance(v,float) else v) for k,v in r.items()} for r in cands[:5]])
best = min(cands, key=lambda r: r["th"]) if cands else max(grid, key=lambda r: r["sens"])
best


Val AUROC: 0.684703196347032
Top (τ with sens≥0.99): [{'th': 0.05, 'tp': 365, 'fn': 0, 'fp': 96, 'tn': 0, 'sens': 1.0, 'spec': 0.0, 'prec': 0.7918, 'acc': 0.7918}, {'th': 0.06, 'tp': 365, 'fn': 0, 'fp': 96, 'tn': 0, 'sens': 1.0, 'spec': 0.0, 'prec': 0.7918, 'acc': 0.7918}, {'th': 0.07, 'tp': 365, 'fn': 0, 'fp': 96, 'tn': 0, 'sens': 1.0, 'spec': 0.0, 'prec': 0.7918, 'acc': 0.7918}, {'th': 0.08, 'tp': 365, 'fn': 0, 'fp': 96, 'tn': 0, 'sens': 1.0, 'spec': 0.0, 'prec': 0.7918, 'acc': 0.7918}, {'th': 0.09, 'tp': 365, 'fn': 0, 'fp': 96, 'tn': 0, 'sens': 1.0, 'spec': 0.0, 'prec': 0.7918, 'acc': 0.7918}]


2025-12-15 09:58:02.655225: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


{'th': 0.05,
 'tp': 365,
 'fn': 0,
 'fp': 96,
 'tn': 0,
 'sens': 0.9999999999972603,
 'spec': 0.0,
 'prec': 0.7917570498898227,
 'acc': 0.7917570498915402}

In [12]:
import pandas as pd
print(pd.Series(val_probs).describe())

count    461.000000
mean       0.408479
std        0.027092
min        0.298739
25%        0.401496
50%        0.419471
75%        0.426320
max        0.439269
dtype: float64


In [13]:
import numpy as np
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve

# get probs/labels from your balanced+focal Stage-2 model
val_probs = np.concatenate([model.predict(x, verbose=0) for x, _ in val_ds]).ravel()
val_labels = np.concatenate([y.numpy() for _, y in val_ds]).astype(int)

print("Val AUROC:", roc_auc_score(val_labels, val_probs))

def metrics_at(th):
    pred = (val_probs >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(val_labels, pred, labels=[0,1]).ravel()
    sens = tp / (tp + fn + 1e-9)   # TPR for Risk
    spec = tn / (tn + fp + 1e-9)
    prec = tp / (tp + fp + 1e-9)
    acc  = (tp + tn) / (tp + tn + fp + fn)
    return dict(th=th, tp=tp, fn=fn, fp=fp, tn=tn, sens=sens, spec=spec, prec=prec, acc=acc)

# Search over all unique score thresholds (plus the edges)
uniq = np.unique(np.sort(val_probs))
grid_th = np.concatenate(([0.0], (uniq[1:]+uniq[:-1])/2.0, [1.0]))
grid = [metrics_at(t) for t in grid_th]

# Feasible set: sensitivity >= 0.99
feasible = [r for r in grid if r["sens"] >= 0.99]
feasible_sorted = sorted(feasible, key=lambda r: (-r["spec"], -r["prec"], r["th"])) if feasible else []

print("Top 5 feasible (maximize specificity, then precision):")
for r in feasible_sorted[:5]:
    print({k:(round(v,4) if isinstance(v,float) else v) for k,v in r.items()})

best = feasible_sorted[0] if feasible_sorted else max(grid, key=lambda r: r["sens"])
best


Val AUROC: 0.684703196347032
Top 5 feasible (maximize specificity, then precision):
{'th': 0.3438, 'tp': 362, 'fn': 3, 'fp': 79, 'tn': 17, 'sens': 0.9918, 'spec': 0.1771, 'prec': 0.8209, 'acc': 0.8221}
{'th': 0.3414, 'tp': 362, 'fn': 3, 'fp': 80, 'tn': 16, 'sens': 0.9918, 'spec': 0.1667, 'prec': 0.819, 'acc': 0.82}
{'th': 0.3396, 'tp': 362, 'fn': 3, 'fp': 81, 'tn': 15, 'sens': 0.9918, 'spec': 0.1562, 'prec': 0.8172, 'acc': 0.8178}
{'th': 0.339, 'tp': 362, 'fn': 3, 'fp': 82, 'tn': 14, 'sens': 0.9918, 'spec': 0.1458, 'prec': 0.8153, 'acc': 0.8156}
{'th': 0.3371, 'tp': 362, 'fn': 3, 'fp': 83, 'tn': 13, 'sens': 0.9918, 'spec': 0.1354, 'prec': 0.8135, 'acc': 0.8134}


{'th': 0.34375691413879395,
 'tp': 362,
 'fn': 3,
 'fp': 79,
 'tn': 17,
 'sens': 0.9917808219150911,
 'spec': 0.1770833333314887,
 'prec': 0.8208616780026738,
 'acc': 0.8221258134490239}

In [14]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

TAU = 0.34375691413879395  # your best τ*
# collect labels/probs from val_ds (reuse model & val_ds already in memory)
val_probs = np.concatenate([model.predict(x, verbose=0) for x,_ in val_ds]).ravel()
val_y     = np.concatenate([y.numpy() for _,y in val_ds]).astype(int)

pred = (val_probs >= TAU).astype(int)
tn, fp, fn, tp = confusion_matrix(val_y, pred, labels=[0,1]).ravel()

sens = tp / (tp + fn + 1e-9)
spec = tn / (tn + fp + 1e-9)
prec = tp / (tp + fp + 1e-9)
acc  = (tp + tn) / (tp + tn + fp + fn)

print(f"Threshold τ = {TAU:.4f}")
print(f"TP={tp}  FN={fn}  FP={fp}  TN={tn}")
print(f"Sensitivity (Recall, Risk=1): {sens:.4f}")
print(f"Specificity (Safe=0):         {spec:.4f}")
print(f"Precision (PPV):               {prec:.4f}")
print(f"Accuracy:                      {acc:.4f}")

print("\nClassification report @τ:")
print(classification_report(val_y, pred, target_names=['Safe(0)','Risk(1)']))


Threshold τ = 0.3438
TP=362  FN=3  FP=79  TN=17
Sensitivity (Recall, Risk=1): 0.9918
Specificity (Safe=0):         0.1771
Precision (PPV):               0.8209
Accuracy:                      0.8221

Classification report @τ:
              precision    recall  f1-score   support

     Safe(0)       0.85      0.18      0.29        96
     Risk(1)       0.82      0.99      0.90       365

    accuracy                           0.82       461
   macro avg       0.84      0.58      0.60       461
weighted avg       0.83      0.82      0.77       461



2025-12-15 10:07:15.870261: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [15]:
def triage_band(p):
    if p >= 0.70: return "RED"
    if p >= 0.3438: return "AMBER"
    if p <  0.20:  return "GREEN"
    return "GREY"  # optional buffer between 0.20–0.3438


In [16]:
import json, os
ART = "data/artifacts"
os.makedirs(ART, exist_ok=True)

# save model (best weights already restored)
model.save(os.path.join(ART, "lesion_safety_bal_focal_best.keras"))

with open(os.path.join(ART, "lesion_safety_config.json"), "w") as f:
    json.dump({"threshold": float(TAU),
               "bands": {"green": 0.20, "amber": float(TAU), "red": 0.70}},
              f, indent=2)
print("Saved model + threshold config.")


Saved model + threshold config.


In [17]:
import tensorflow as tf
inp  = tf.keras.Input(shape=(224,224,3), name="image_rgb_0_255")
x    = tf.keras.layers.Rescaling(1./127.5, offset=-1)(inp)
base = tf.keras.applications.MobileNetV3Small(include_top=False, weights=None, pooling="avg")
base.set_weights(model.get_layer("MobileNetV3Small").get_weights())
x    = base(x, training=False)
x    = tf.keras.layers.Dropout(0.40)(x)
p    = tf.keras.layers.Dense(1, activation="sigmoid", name="risk")(x)
out  = tf.keras.layers.Lambda(lambda z: tf.cast(z >= TAU, tf.float32), name="triage")(p)
wrapped = tf.keras.Model(inp, [p, out], name="LesionSafetyWithThreshold")
wrapped.save(os.path.join(ART, "lesion_safety_with_tau.keras"))
print("Saved wrapped model with threshold output.")


Saved wrapped model with threshold output.


In [18]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)  # or 'wrapped'
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # dynamic-range quant
tflite_model = converter.convert()
open(os.path.join(ART, "lesion_safety_bal_focal_best.tflite"), "wb").write(tflite_model)
print("TFLite saved.")


INFO:tensorflow:Assets written to: /tmp/tmpjwmbyohr/assets


INFO:tensorflow:Assets written to: /tmp/tmpjwmbyohr/assets


Saved artifact at '/tmp/tmpjwmbyohr'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_186')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  134613094461712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134613094462288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134613094462672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134613094462096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134613094461520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134613094463248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134613094464592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134613094464784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134613094464400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134613094463632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1346130944

W0000 00:00:1765793384.858642  116260 tf_tfl_flatbuffer_helpers.cc:392] Ignored output_format.
W0000 00:00:1765793384.858700  116260 tf_tfl_flatbuffer_helpers.cc:395] Ignored drop_control_dependency.
2025-12-15 10:09:44.859312: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpjwmbyohr
2025-12-15 10:09:44.867556: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2025-12-15 10:09:44.867588: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpjwmbyohr
2025-12-15 10:09:44.967408: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2025-12-15 10:09:44.978509: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2025-12-15 10:09:45.446006: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpjwmbyohr
2025-12-15 10:09:45.555250: I tensorflow/cc/saved_model/loader.cc

In [20]:
# --- TFLite inference (dynamic batch safe) ---
interpreter = tf.lite.Interpreter(model_path=TFL_PATH)
interpreter.allocate_tensors()

inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

# Cast to expected dtype (often float32 after dynamic-range quant)
xb = batch.astype(inp["dtype"])

# Try to use dynamic shape if supported; otherwise, resize explicitly to (N,224,224,3)
N = xb.shape[0]
shape_sig = inp.get("shape_signature", None)
can_dynamic = shape_sig is not None and (-1 in list(shape_sig))

if can_dynamic or (tuple(inp["shape"]) != xb.shape):
    interpreter.resize_tensor_input(inp["index"], [N, 224, 224, 3])
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]

# Inference
interpreter.set_tensor(inp["index"], xb)
interpreter.invoke()
probs_tfl = interpreter.get_tensor(out["index"]).reshape(-1)

# Compare with Keras
mae = float(np.mean(np.abs(probs_keras - probs_tfl)))
corr = float(np.corrcoef(probs_keras, probs_tfl)[0,1])
print(f"TFLite vs Keras — MAE={mae:.6f}, Pearson r={corr:.4f}")
print("Example pairs:", list(zip(np.round(probs_keras[:5],4), np.round(probs_tfl[:5],4))))


TFLite vs Keras — MAE=0.062013, Pearson r=-0.0163
Example pairs: [(0.4156, 0.4581), (0.4354, 0.4553), (0.4221, 0.49), (0.4246, 0.486), (0.4258, 0.4783)]


In [21]:
# TFLite vs Keras consistency check on val set
import numpy as np, pandas as pd, tensorflow as tf, os, random
from PIL import Image

ART = "data/artifacts"
TFL_PATH = os.path.join(ART, "lesion_safety_bal_focal_best.tflite")   # from Step 3
KERAS_PATH = os.path.join(ART, "lesion_safety_bal_focal_best.keras")  # from Step 3

# Load val_df and sample a few images
N = 32
subset = val_df.sample(n=min(N, len(val_df)), random_state=7)

def load_rgb_uint8(path, size=(224,224)):
    img = Image.open(path).convert("RGB").resize(size, Image.BILINEAR)
    return np.array(img, dtype=np.uint8)

# Stack a batch
batch = np.stack([load_rgb_uint8(p) for p in subset["fullpath"]], axis=0)

# Keras model (expects 0–255; internal Rescaling layer handles normalization)
keras_model = tf.keras.models.load_model(KERAS_PATH, compile=False)
probs_keras = keras_model.predict(batch, verbose=0).ravel()

# TFLite inference
interpreter = tf.lite.Interpreter(model_path=TFL_PATH)
interpreter.allocate_tensors()
inp_details  = interpreter.get_input_details()[0]
out_details  = interpreter.get_output_details()[0]

# Ensure dtype/shape match
x = batch.astype(np.float32) if inp_details["dtype"]==np.float32 else batch
if tuple(inp_details["shape"]) != x.shape:
    x = x.astype(inp_details["dtype"])
    x = x.reshape(inp_details["shape"])

# Run
interpreter.set_tensor(inp_details["index"], x)
interpreter.invoke()
probs_tfl = interpreter.get_tensor(out_details["index"]).reshape(-1)

# Compare
mae = float(np.mean(np.abs(probs_keras - probs_tfl)))
corr = float(np.corrcoef(probs_keras, probs_tfl)[0,1])
print(f"TFLite vs Keras — MAE={mae:.6f}, Pearson r={corr:.4f}")
print("Example pairs:", list(zip(np.round(probs_keras[:5],4), np.round(probs_tfl[:5],4))))


: 

In [1]:
import os, numpy as np, tensorflow as tf
from PIL import Image

# 1) Safer TFLite runtime (must be set BEFORE creating the interpreter)
os.environ["TF_LITE_DISABLE_XNNPACK"] = "1"

ART = "data/artifacts"
TFL_PATH   = f"{ART}/lesion_safety_bal_focal_best.tflite"
KERAS_PATH = f"{ART}/lesion_safety_bal_focal_best.keras"

# 2) Load a small batch from your existing val_df (adjust N if desired)
N = 16
subset = val_df.sample(n=min(N, len(val_df)), random_state=7)

def load_rgb_uint8(path, size=(224,224)):
    img = Image.open(path).convert("RGB").resize(size, Image.BILINEAR)
    return np.array(img, dtype=np.uint8)

batch = np.stack([load_rgb_uint8(p) for p in subset["fullpath"]], axis=0)

# 3) Keras inference (model expects 0–255; it handles rescaling internally)
keras_model = tf.keras.models.load_model(KERAS_PATH, compile=False)
probs_keras = keras_model.predict(batch, verbose=0).ravel()

# 4) TFLite — per-sample loop, fixed (1,224,224,3) input
interpreter = tf.lite.Interpreter(model_path=TFL_PATH)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

probs_tfl = []
for i in range(batch.shape[0]):
    xi = batch[i:i+1]
    xi = xi.astype(inp["dtype"])  # usually float32 after dynamic-range quant
    interpreter.set_tensor(inp["index"], xi)
    interpreter.invoke()
    probs_tfl.append(float(interpreter.get_tensor(out["index"]).ravel()[0]))
probs_tfl = np.array(probs_tfl)

# 5) Consistency metrics
mae = float(np.mean(np.abs(probs_keras - probs_tfl)))
corr = float(np.corrcoef(probs_keras, probs_tfl)[0,1])
print(f"TFLite vs Keras — MAE={mae:.6f}, Pearson r={corr:.4f}")
print("Example pairs:", list(zip(np.round(probs_keras[:5],4), np.round(probs_tfl[:5],4))))


2025-12-15 11:19:21.258210: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-15 11:19:21.640679: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-15 11:19:21.779364: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-15 11:19:21.825318: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-15 11:19:22.104059: I tensorflow/core/platform/cpu_feature_guar

NameError: name 'val_df' is not defined